# Model testing

Loads each of the 6 policies saved by `train_models.ipynb` from `saved_policies/<name>/` and checks the success criterion: **100 consecutive greedy evaluation episodes each surviving to 600 simulation ticks**. Run cells individually per algorithm, or run the whole notebook top to bottom to build the summary table at the end.

In [ ]:
import json
import os

from cart_model import Cart_model
from discrete_policies import SarsaAgent, QLearningAgent, ExpectedSarsaAgent
from continuous_policies import ActorPolicyContinuousSpace, ReinforcePolicy, SarsaTileCoding
from train import evaluate_success_criterion, evaluate_average_reward

context = Cart_model(human=False)
results = {}

## SARSA (discrete)

In [ ]:
sarsa_policy = SarsaAgent.load("saved_policies/sarsa")
results["sarsa"] = evaluate_success_criterion(sarsa_policy, context, required_streak=100, max_steps=600)
print("sarsa", results["sarsa"])

## Q-learning (discrete)

In [ ]:
q_learning_policy = QLearningAgent.load("saved_policies/q_learning")
results["q_learning"] = evaluate_success_criterion(q_learning_policy, context, required_streak=100, max_steps=600)
print("q_learning", results["q_learning"])

## Expected SARSA (discrete)

In [ ]:
expected_sarsa_policy = ExpectedSarsaAgent.load("saved_policies/expected_sarsa")
results["expected_sarsa"] = evaluate_success_criterion(expected_sarsa_policy, context, required_streak=100, max_steps=600)
print("expected_sarsa", results["expected_sarsa"])

## Actor-Critic (continuous)

In [ ]:
actor_critic_policy = ActorPolicyContinuousSpace.load("saved_policies/actor_critic")
results["actor_critic"] = evaluate_success_criterion(actor_critic_policy, context, required_streak=100, max_steps=600)
print("actor_critic", results["actor_critic"])

## REINFORCE (continuous)

In [ ]:
reinforce_policy = ReinforcePolicy.load("saved_policies/reinforce")
results["reinforce"] = evaluate_success_criterion(reinforce_policy, context, required_streak=100, max_steps=600)
print("reinforce", results["reinforce"])

## Semi-gradient SARSA with tile coding (continuous state, discretized action)

In [ ]:
sarsa_tile_coding_policy = SarsaTileCoding.load("saved_policies/sarsa_tile_coding")
results["sarsa_tile_coding"] = evaluate_success_criterion(sarsa_tile_coding_policy, context, required_streak=100, max_steps=600)
print("sarsa_tile_coding", results["sarsa_tile_coding"])

## Summary

`met=True` means the policy reached 100 consecutive 600-tick survivals. Otherwise `longest_streak` and `mean_survival` (in ticks) show how close it got.

In [ ]:
for name, report in results.items():
    print(f"{name:20s} met={report['met']!s:5s} longest_streak={report['longest_streak']:4d} "
          f"mean_survival={report['mean_survival']:.1f} episodes_run={report['episodes_run']}")

os.makedirs("results", exist_ok=True)
with open("results/summary.json", "w") as f:
    json.dump(results, f, indent=2)

## Watch a policy run (optional, renders a window)

Set `policy` to whichever trained policy you want to watch, then run the cell. Uses a fresh `Cart_model(human=True)` so it doesn't disturb the headless `context` used above.

In [ ]:
from cart_model import Episode

policy = sarsa_policy  # change to whichever policy you want to watch

render_context = Cart_model(human=True)
episode = Episode()
episode.runSlow(render_context, policy=policy, max_steps=600, delay=0.02)
print("survived", len(episode.interactions), "steps, terminated:", episode.terminated, "truncated:", episode.truncated)
render_context.close()

In [ ]:
context.close()